# Data Preparation for EDA and clustering steps

In [15]:
import importlib
import sys, os

# Ensure project root is on path
sys.path.insert(0, os.path.abspath(".."))

import src.code.class_pipeline_functions as cpf
import src.code.io_utils as io

importlib.reload(cpf)
importlib.reload(io)

import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.code.class_pipeline_functions import ClientFeatureEngineer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.3f}".format)

# Path 
ABT_OUT_PATH  = "../data/prepared/abt.parquet"

In [2]:
customer = io.load(ABT_OUT_PATH)
customer.head(5)

[LOAD] ../data/prepared/abt.parquet | shape: (148729, 88)


,CONTRIB,N_CONTRACTS,FIRST_DCREAT,LAST_DCREAT,LAST_DPOS,LAST_DATFIN,FIRST_D1FIN,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,TOTAL_MENSALIDADE,TOTAL_CRD,TOTAL_SREC,TOTAL_RN,TOTAL_RD,LAST_RISK,MAX_RISKA,MIN_RESSO,MAX_RESSO,MEDIAN_RESSO,CSP,NBENF,...,MONTVENC_CP_MEDIAN,MONTVENC_AUTO_MIN,MONTVENC_AUTO_MAX,MONTVENC_AUTO_MEDIAN,MONTVENC_HT_MIN,MONTVENC_HT_MAX,MONTVENC_HT_MEDIAN,DIVIDAS_CL_MIN,DIVIDAS_CL_MAX,DIVIDAS_CL_MEDIAN,DIVIDAS_CP_MIN,DIVIDAS_CP_MAX,DIVIDAS_CP_MEDIAN,DIVIDAS_AUTO_MIN,DIVIDAS_AUTO_MAX,DIVIDAS_AUTO_MEDIAN,DIVIDAS_HT_MIN,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,sys_data_procura,kp_sqe_enc,ks_score_tier,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_SITFAM,sdem_HABITAT,sdem_age
0,00008246f87bcc3c17b90629bb183fe2e58795176310f0...,2,2024-06-25,2024-09-13,2025-11-17,2024-09-30,2024-06-27,84.000,120.000,102.000,3.000,13.000,8.000,19.000,19.000,19.000,24000.000,24000.000,438.761,0.000,0.000,0.000,0.000,0,0.000,1513.466,1513.466,1513.466,80.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,8000.000,0.000,1.650,3701.730,978.250,0.000,58633.860,15934.050,0.000,0.000,0.000,2025-02-05,7.000,2.000,10.000,1.000,2.000,-438.900,10.000,S,F,33.000
1,0000ab2116257783438c70ff85a3e98f2d4194ebe53434...,1,2018-03-29,2018-03-29,2018-04-16,2018-04-16,2018-04-16,120.000,120.000,120.000,91.000,91.000,91.000,91.000,91.000,91.000,20000.000,20000.000,347.447,8115.247,0.000,0.000,0.000,0,0.000,1113.258,1113.258,1113.258,80.000,1.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,20308.940,69754.560,24983.220,5499.880,21274.830,7115.460,0.000,0.000,0.000,0.000,0.000,0.000,2025-04-09,3.000,1.000,10.000,1.000,0.000,1166.900,10.000,C,P,52.000
2,0000c74654405ec1da4dbdcd00b86e397954043965d98e...,2,2001-09-21,2001-09-27,2014-06-27,2001-10-04,2001-09-21,60.000,60.000,60.000,21.000,21.000,21.000,21.000,21.000,21.000,19453.110,19453.110,608.914,0.000,0.000,2.000,24.000,-9223372036854775808,0.000,678.483,678.483,678.483,60.000,2.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN
3,0000e359b15a80ba12c7f60c0fe06ffc68621c87f13a4f...,1,2019-01-28,2019-01-28,2022-12-28,2019-02-04,2019-02-04,72.000,72.000,72.000,34.000,34.000,34.000,34.000,34.000,34.000,2500.000,2500.000,56.018,0.000,0.000,1.000,1.000,210210000000,0.000,838.186,838.186,838.186,91.000,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,3.000,0.000,0.000,590.333,3.000,S,A,69.000
4,0000f858346061c53064586a3347b34659565a6712d004...,1,2019-09-23,2019-09-23,2019-09-30,2019-09-30,2019-09-30,84.000,84.000,84.000,74.000,74.000,74.000,74.000,74.000,74.000,5000.000,5000.000,100.074,883.500,0.000,0.000,0.000,0,0.000,1314.144,1314.144,1314.144,80.000,2.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1019.130,2957.280,2114.780,1465.730,3475.320,1743.460,773.460,2408.560,1695.620,83852.710,168563.040,84906.010,NaT,NaN,NaN,4.000,1.000,1.000,1358.250,4.000,U,A,39.000


In [5]:
customer.describe()

,N_CONTRACTS,FIRST_DCREAT,LAST_DCREAT,LAST_DPOS,LAST_DATFIN,FIRST_D1FIN,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,TOTAL_MENSALIDADE,TOTAL_CRD,TOTAL_SREC,TOTAL_RN,TOTAL_RD,LAST_RISK,MAX_RISKA,MIN_RESSO,MAX_RESSO,MEDIAN_RESSO,CSP,NBENF,EVER_SOL,...,MONTVENC_CP_MIN,MONTVENC_CP_MAX,MONTVENC_CP_MEDIAN,MONTVENC_AUTO_MIN,MONTVENC_AUTO_MAX,MONTVENC_AUTO_MEDIAN,MONTVENC_HT_MIN,MONTVENC_HT_MAX,MONTVENC_HT_MEDIAN,DIVIDAS_CL_MIN,DIVIDAS_CL_MAX,DIVIDAS_CL_MEDIAN,DIVIDAS_CP_MIN,DIVIDAS_CP_MAX,DIVIDAS_CP_MEDIAN,DIVIDAS_AUTO_MIN,DIVIDAS_AUTO_MAX,DIVIDAS_AUTO_MEDIAN,DIVIDAS_HT_MIN,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,sys_data_procura,kp_sqe_enc,ks_score_tier,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_age
count,148729.000,148729,148729,148729,148729,148729,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,...,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,62173,62173.000,62173.000,141115.000,141115.000,141115.000,141115.000,141115.000,141115.000
mean,1.248,2021-11-26 00:55:05.448163584,2022-05-13 12:16:22.630152704,2024-01-03 17:54:06.363520256,2022-05-22 06:24:52.825205504,2021-12-04 06:36:57.235374080,69.853,74.417,72.129,30.974,35.070,32.889,43.978,44.207,44.110,11521.808,11521.624,262.458,5917.065,-6.465,0.340,1.134,-683574328200651648.000,0.083,1443.922,1493.280,1469.255,62.417,0.612,0.183,...,4.725,207.192,30.687,4.977,83.138,13.782,4.985,147.024,28.034,6795.433,22142.234,10722.432,1248.404,5200.607,2400.276,3112.368,10089.306,5114.468,24906.039,57096.301,29331.216,2024-08-28 00:30:14.910009344,2.152,1.572,8.570,0.782,0.683,438.160,8.570,45.050
min,1.000,1996-06-14 00:00:00,1996-06-26 00:00:00,2014-06-27 00:00:00,1996-07-03 00:00:00,1996-06-14 00:00:00,11.000,11.000,11.000,0.000,0.000,0.000,0.000,0.000,0.000,300.000,300.000,0.621,0.000,-2641.014,0.000,0.000,-9223372036854775808.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2020-08-12 00:00:00,0.000,1.000,1.000,0.000,0.000,-2382.500,1.000,18.000
25%,1.000,2020-06-29 00:00:00,2020-12-22 00:00:00,2023-08-31 00:00:00,2020-12-30 00:00:00,2020-07-07 00:00:00,48.000,48.000,48.000,9.000,14.000,12.000,22.000,22.000,22.000,4900.000,4900.000,115.076,0.000,0.000,0.000,0.000,0.000,0.000,948.935,977.381,963.831,55.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3500.000,1157.435,0.000,724.565,129.830,0.000,0.000,0.000,0.000,0.000,0.000,2023-12-16 00:00:00,0.000,1.000,4.000,0.000,0.000,-716.750,4.000,36.000
50%,1.000,2022-08-17 00:00:00,2023-04-03 00:00:00,2024-10-02 00:00:00,2023-04-11 00:00:00,2022-08-26 00:00:00,81.000,84.000,84.000,24.000,30.000,27.000,39.000,39.000,39.000,7634.360,7633.110,187.685,2800.762,0.000,0.000,0.000,0.000,0.000,1214.224,1247.041,1230.628,70.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2445.450,10934.160,5946.660,195.250,2296.270,1076.080,0.000,1175.990,0.000,0.000,0.000,0.000,2024-10-11 00:00:00,1.000,1.000,7.000,1.000,1.000,253.786,7.000,44.000
75%,1.000,2024-04-06 00:00:00,2024-11-15 00:00:00,2025-07-01 00:00:00,2024-11-22 00:00:00,2024-04-11 00:00:00,84.000,84.000,84.000,48.000,52.000,48.000,61.000,61.000,61.000,15000.000,15000.000,323.692,8179.504,0.000,0.000,0.000,0.000,0.000,1713.806,1747.056,1731.813,80.000,1.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,9121.730,25932.710,14665.720,1300.885,6118.600,3010.030,3265.190,14000.000,8

## 1. Missing values analysis

In [3]:
# Check for columns with >= 30% missing values
null_pct = (customer.isnull().sum() / len(customer) * 100).round(2)
high_missing = null_pct[null_pct >= 30].sort_values(ascending=False)

print(f"Columns with >= 30% missing values: {len(high_missing)}\n")
if len(high_missing) > 0:
    display(high_missing.to_frame("null_%"))
else:
    print("None found. All columns have < 30% missing.")

Columns with >= 30% missing values: 6



,null_%
LAST_OBS_DATE_RBT,97.900
LAST_OBS_DATE_SOL,81.670
LAST_OBS_DATE_SAN,72.700
sys_data_procura,58.200
kp_sqe_enc,58.200
ks_score_tier,58.200


In [4]:
# Drop columns with >= 30% missing values
cols_to_drop = ['LAST_OBS_DATE_RBT', 'LAST_OBS_DATE_SOL', 'LAST_OBS_DATE_SAN',
                'sys_data_procura', 'kp_sqe_enc', 'ks_score_tier']

customer.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"New shape: {customer.shape}")

Dropped 6 columns: ['LAST_OBS_DATE_RBT', 'LAST_OBS_DATE_SOL', 'LAST_OBS_DATE_SAN', 'sys_data_procura', 'kp_sqe_enc', 'ks_score_tier']
New shape: (148729, 82)


In [5]:
# Drop rows with > 50% missing values
threshold = 0.5
min_non_null = int(threshold * customer.shape[1])  # minimum non-null values required
rows_before = customer.shape[0]
customer.dropna(thresh=min_non_null, inplace=True)
rows_after = customer.shape[0]
print(f"Dropped {rows_before - rows_after} rows with > 50% missing values")
print(f"New shape: {customer.shape}")

Dropped 1314 rows with > 50% missing values
New shape: (147415, 82)


## 2. Converting date time features to numeric features

In [6]:
date_cols = customer.select_dtypes(include=['datetime64', 'datetimetz']).columns.tolist()
print(f"Date columns ({len(date_cols)}):\n")
for col in date_cols:
    print(f"  - {col}")

Date columns (5):

  - FIRST_DCREAT
  - LAST_DCREAT
  - LAST_DPOS
  - LAST_DATFIN
  - FIRST_D1FIN


In [7]:
ref_date = pd.Timestamp('2025-11-30')

# Total client seniority (years)
customer['CLIENT_SENIORITY_YEARS'] = (ref_date - customer['FIRST_DCREAT']).dt.days / 365.25

# Months since last contract (recency)
customer['MONTHS_SINCE_LAST_CONTRACT'] = (ref_date - customer['LAST_DCREAT']).dt.days / 30

# Months remaining until end of current contract (can be negative, represents how many time has passed since )
customer['MONTHS_TO_CONTRACT_END'] = (customer['LAST_DATFIN'] - ref_date).dt.days / 30

# Current contract duration (months)
customer['CURRENT_CONTRACT_DURATION_MONTHS'] = (customer['LAST_DATFIN'] - customer['LAST_DCREAT']).dt.days / 30

# First contract duration (historical profile)
customer['FIRST_CONTRACT_DURATION_MONTHS'] = (customer['FIRST_D1FIN'] - customer['FIRST_DCREAT']).dt.days / 30

# Years between first and last contract (repeat behaviour)
customer['YEARS_BETWEEN_FIRST_AND_LAST'] = (customer['LAST_DCREAT'] - customer['FIRST_DCREAT']).dt.days / 365.25

In [ ]:
#customer = customer.drop(columns=date_cols)
print(f"Shape: {customer.shape}")

Shape: (147415, 83)


## 3. Imputation

In [9]:
# Identify column types
cat_cols = customer.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = customer.select_dtypes(include=['number']).columns.tolist()

In [10]:
# 1. Impute categoricals with 'Unknown'
customer[cat_cols] = customer[cat_cols].fillna('Unknown')
# 2. Impute numericals with IterativeImputer
imputer = IterativeImputer(max_iter=10, random_state=42)
customer[num_cols] = imputer.fit_transform(customer[num_cols])
print(f"Remaining nulls: {customer.isnull().sum().sum()}")
print(f"Shape: {customer.shape}")


Remaining nulls: 0
Shape: (147415, 83)


In [20]:
customer.describe()

,N_CONTRACTS,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,TOTAL_MENSALIDADE,TOTAL_CRD,TOTAL_SREC,TOTAL_RN,TOTAL_RD,LAST_RISK,MAX_RISKA,MIN_RESSO,MAX_RESSO,MEDIAN_RESSO,CSP,NBENF,EVER_SOL,N_SOL,EVER_SAN,N_SAN,EVER_RBT,N_RBT,...,MONTVENC_AUTO_MIN,MONTVENC_AUTO_MAX,MONTVENC_AUTO_MEDIAN,MONTVENC_HT_MIN,MONTVENC_HT_MAX,MONTVENC_HT_MEDIAN,DIVIDAS_CL_MIN,DIVIDAS_CL_MAX,DIVIDAS_CL_MEDIAN,DIVIDAS_CP_MIN,DIVIDAS_CP_MAX,DIVIDAS_CP_MEDIAN,DIVIDAS_AUTO_MIN,DIVIDAS_AUTO_MAX,DIVIDAS_AUTO_MEDIAN,DIVIDAS_HT_MIN,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_age,CLIENT_SENIORITY_YEARS,MONTHS_SINCE_LAST_CONTRACT,MONTHS_TO_CONTRACT_END,CURRENT_CONTRACT_DURATION_MONTHS,FIRST_CONTRACT_DURATION_MONTHS,YEARS_BETWEEN_FIRST_AND_LAST
count,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,...,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000,147415.000
mean,1.250,69.921,74.522,72.215,31.026,35.155,32.956,44.114,44.345,44.247,11552.335,11552.150,262.809,5969.807,-6.523,0.341,1.124,-608638318002486656.000,0.083,1446.580,1496.379,1472.139,62.426,0.613,0.176,0.182,0.275,0.331,0.021,0.022,...,-32.512,103.207,-18.019,-149.541,-1014.745,-959.621,6379.704,29172.596,17951.966,3304.791,13794.446,6104.567,-18396.096,3846.048,-85158.975,266325.269,134655.999,-54237.617,8.643,0.751,0.692,435.389,8.630,45.614,3.873,41.484,-41.193,0.291,0.277,0.465
std,0.609,29.980,29.655,29.086,25.848,25.510,25.053,28.992,28.972,28.970,11472.087,11472.004,239.355,8636.782,240.344,0.891,3.360,2307375226187280384.000,0.548,5555.095,5558.611,5555.892,20.804,0.868,0.381,0.403,0.447,0.607,0.144,0.155,...,560.179,1498.471,611.903,1542.858,30830.668,19663.728,26942.750,207918.466,127638.218,32257.585,168895.592,62976.630,464659.268,130724.859,2001729.388,5372439.661,1828122.319,1848264.942,7.083,1.118,0.798,1663.234,7.886,31.908,3.254,40.288,40.221,0.353,0.345,1.293
min,1.000,11.000,11.000,11.000,0.000,0.000,0.000,0.000,0.000,0.000,300.000,300.000,0.621,0.000,-2641.014,0.000,0.000,-9223372036854775808.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,-3362.893,-177089.573,-3017.203,-12595.174,-146622.106,-105288.212,-87396.668,-72532371.594,-41117208.816,-9886729.573,-55896711.552,-20058853.402,-2619913.349,-733782.367,-11136923.486,-1821502584.357,-624401597.027,-10262855.257,-357.056,-240.869,-21.396,-57177.893,-236.284,-1077.147,-0.005,-0.067,-339.467,0.000,0.000,0.000
25%,1.000,48.000,48.000,48.000,9.000,14.000,12.000,22.000,23.000,22.000,4900.000,4900.000,114.934,0.000,0.000,0.000,0.000,0.000,0.000,951.643,977.381,967.895,55.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3500.000,1181.825,0.000,727.320,133.320,0.000,0.000,0.000,0.000,0.000,0.000,4.000,0.000,0.000,-712.667,4.000,36.000,1.640,12.500,-58.667,0.067,0.067,0.000
50%,1.000,84.000,84.000,84.000,24.000,30.000,27.000,39.000,40.000,40.000,7700.000,7700.000,187.729,2857.358,0.000,0.000,0.000,0.000,0.000,1215.591,1249.777,1233.363,70.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,2324.130,11384.180,6218.680,225.240,2405.350,1136.260,0.000,426.870,0.000,0.000,0.000,0.000,7.000,1.000,1.000,250.000,7.000,44.000,3.264,32.067,-31.800,0.167,0.167,0.000
75%,1.000,84.000,84.000,84.000,48.000,53.000,48.000,61.000,61.000,61

## 4. Encoding categorical features

In [11]:
customer[cat_cols]

,CONTRIB,sdem_SITFAM,sdem_HABITAT
0,00008246f87bcc3c17b90629bb183fe2e58795176310f0...,S,F
1,0000ab2116257783438c70ff85a3e98f2d4194ebe53434...,C,P
3,0000e359b15a80ba12c7f60c0fe06ffc68621c87f13a4f...,S,A
4,0000f858346061c53064586a3347b34659565a6712d004...,U,A
5,00025459b703e1c308553e83a6d545a71fe6a787c2dd1c...,C,P
...,...,...,...
148724,fffc991d73df732084dab58938d520b8a5d8712474fa53...,S,P
148725,fffd54b6dbb4cc001fb8ab52a905cff9bdbf14b747be9a...,U,F
148726,fffe8c5cbb44ad3ec6154ef5a5208ecd76594a88377f00...,X,F
148727,ffff943c736f98d4840f65328ba372a29689312dd781b4...,Unknown,Unknown


In [12]:
customer.drop(columns=['CONTRIB'], inplace=True) #unique for each customer, need to drop it now because we won't encode it
cat_cols.remove('CONTRIB')

In [13]:
# One-Hot Encode 
customer = pd.get_dummies(customer, columns=['sdem_SITFAM', 'sdem_HABITAT'], drop_first=True, dtype=int)
print(f"Shape after encoding: {customer.shape}")
customer.head()

Shape after encoding: (147415, 96)


,N_CONTRACTS,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,TOTAL_MENSALIDADE,TOTAL_CRD,TOTAL_SREC,TOTAL_RN,TOTAL_RD,LAST_RISK,MAX_RISKA,MIN_RESSO,MAX_RESSO,MEDIAN_RESSO,CSP,NBENF,EVER_SOL,N_SOL,EVER_SAN,N_SAN,EVER_RBT,N_RBT,...,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_age,CLIENT_SENIORITY_YEARS,MONTHS_SINCE_LAST_CONTRACT,MONTHS_TO_CONTRACT_END,CURRENT_CONTRACT_DURATION_MONTHS,FIRST_CONTRACT_DURATION_MONTHS,YEARS_BETWEEN_FIRST_AND_LAST,sdem_SITFAM_D,sdem_SITFAM_F,sdem_SITFAM_P,sdem_SITFAM_S,sdem_SITFAM_U,sdem_SITFAM_Unknown,sdem_SITFAM_V,sdem_SITFAM_X,sdem_HABITAT_A,sdem_HABITAT_E,sdem_HABITAT_F,sdem_HABITAT_L,sdem_HABITAT_O,sdem_HABITAT_P,sdem_HABITAT_Unknown,sdem_HABITAT_X
0,2.000,84.000,120.000,102.000,3.000,13.000,8.000,19.000,19.000,19.000,24000.000,24000.000,438.761,0.000,0.000,0.000,0.000,0.000,0.000,1513.466,1513.466,1513.466,80.000,0.000,0.000,0.000,1.000,2.000,0.000,0.000,...,0.000,0.000,10.000,1.000,2.000,-438.900,10.000,33.000,1.432,14.767,-14.200,0.567,0.067,0.219,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0
1,1.000,120.000,120.000,120.000,91.000,91.000,91.000,91.000,91.000,91.000,20000.000,20000.000,347.447,8115.247,0.000,0.000,0.000,0.000,0.000,1113.258,1113.258,1113.258,80.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,10.000,1.000,0.000,1166.900,10.000,52.000,7.674,93.433,-92.833,0.600,0.600,0.000,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,1.000,72.000,72.000,72.000,34.000,34.000,34.000,34.000,34.000,34.000,2500.000,2500.000,56.018,0.000,0.000,1.000,1.000,210210000000.000,0.000,838.186,838.186,838.186,91.000,0.000,0.000,0.000,1.000,1.000,0.000,0.000,...,1744393.976,-1703195.032,3.000,0.000,0.000,590.333,3.000,69.000,6.839,83.267,-83.033,0.233,0.233,0.000,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0
4,1.000,84.000,84.000,84.000,74.000,74.000,74.000,74.000,74.000,74.000,5000.000,5000.000,100.074,883.500,0.000,0.000,0.000,0.000,0.000,1314.144,1314.144,1314.144,80.000,2.000,0.000,0.000,0.000,0.000,0.000,0.000,...,168563.040,84906.010,4.000,1.000,1.000,1358.250,4.000,39.000,6.188,75.333,-75.100,0.233,0.233,0.000,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0
5,1.000,60.000,60.000,60.000,35.000,35.000,35.000,37.000,37.000,37.000,6000.000,6000.000,162.098,3419.266,0.000,0.000,0.000,0.000,0.000,1031.650,1031.650,1031.650,80.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,17.000,1.000,1.000,529.000,17.000,55.000,2.891,35.200,-35.067,0.133,0.133,0.000,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


## 5. Feature Engineering

In [ ]:
# Apply feature engineering
fe = ClientFeatureEngineer(
    include_composite=True,
    drop_date_cols=True,   # drops raw date columns (DPOS, DCREAT, etc.)
    verbose=True           # prints summary of features created
)
customer = fe.fit_transform(customer)
print(f"Shape after feature engineering: {customer.shape}")
customer.head()

In [16]:
expected = ['MTFINO', 'CRD', 'DPOS', 'DCREAT', 'MENSALIDADE', 'RESSO', 'RISK']
missing = [c for c in expected if c not in customer.columns]
print("Missing expected columns:", missing)

Missing expected columns: ['MTFINO', 'CRD', 'DPOS', 'DCREAT', 'MENSALIDADE', 'RESSO', 'RISK']
